In [1]:
import pandas as pd
import hashlib
import json
from pathlib import Path
from datetime import datetime

In [3]:
# import shutil
# from pathlib import Path
# from datetime import datetime

# # Paths
# HISTORY_PATH = Path("selection_history.json")
# TRAIN_DIR    = Path("training_sets")
# OUTPUT_DIR   = Path(".")

# # Backup folder with timestamp
# BACKUP_DIR = Path("backup_before_reset") / datetime.now().strftime("%Y%m%d_%H%M%S")
# BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# # Move history file if present
# if HISTORY_PATH.exists():
#     shutil.move(str(HISTORY_PATH), BACKUP_DIR / HISTORY_PATH.name)

# # Move previous training sets
# if TRAIN_DIR.exists():
#     shutil.move(str(TRAIN_DIR), BACKUP_DIR / TRAIN_DIR.name)

# # Move any unique_sample_*.csv files
# moved_any = False
# for p in OUTPUT_DIR.glob("unique_sample_*.csv"):
#     shutil.move(str(p), BACKUP_DIR / p.name)
#     moved_any = True

# print("✅ Reset complete. Archived prior state to:", BACKUP_DIR.resolve())

import shutil
from pathlib import Path
from datetime import datetime

# Paths (fixed leading slashes)
HISTORY_PATH = Path("/home/ubuntu/TW_MultiLabel_SMP/jupyter-notebooks/selection_history.json")
TRAIN_DIR    = Path("/home/ubuntu/TW_MultiLabel_SMP/datasets/training_sets")
OUTPUT_DIR   = Path(".")

# Backup folder with timestamp
BACKUP_DIR = Path("/home/ubuntu/TW_MultiLabel_SMP/datasets/backup_before_reset") / datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# ---- Preserve history (copy, don't move) ----
if HISTORY_PATH.exists():
    backup_history = BACKUP_DIR / HISTORY_PATH.name
    try:
        shutil.copy2(HISTORY_PATH, backup_history)
        print(f"📝 Preserved history: copied to {backup_history}")
    except Exception as e:
        print(f"⚠️ Could not copy history file: {e}")
else:
    print("ℹ️ No selection_history.json found to preserve.")

# ---- Move previous training sets to backup ----
if TRAIN_DIR.exists():
    dest = BACKUP_DIR / TRAIN_DIR.name
    try:
        shutil.move(str(TRAIN_DIR), dest)
        print(f"📦 Archived training_sets to: {dest}")
    except Exception as e:
        print(f"⚠️ Could not move training_sets: {e}")
else:
    print("ℹ️ No training_sets directory found to archive.")

# ---- Move any unique_sample_*.csv files to backup ----
moved_any = False
for p in OUTPUT_DIR.glob("unique_sample_*.csv"):
    try:
        shutil.move(str(p), BACKUP_DIR / p.name)
        print(f"📄 Archived {p.name}")
        moved_any = True
    except Exception as e:
        print(f"⚠️ Could not move {p}: {e}")

if not moved_any:
    print("ℹ️ No unique_sample_*.csv files found to archive.")

print("✅ Reset complete. Old artifacts archived. History preserved in place.")



📝 Preserved history: copied to /home/ubuntu/TW_MultiLabel_SMP/datasets/backup_before_reset/20251016_174855/selection_history.json
ℹ️ No training_sets directory found to archive.
ℹ️ No unique_sample_*.csv files found to archive.
✅ Reset complete. Old artifacts archived. History preserved in place.


In [4]:
def row_hash(row: pd.Series) -> str:
    """Generate a deterministic hash of a row if no explicit ID column exists."""
    obj = row.to_dict()
    normalized = {str(k): ("" if pd.isna(v) else str(v)) for k, v in obj.items()}
    payload = json.dumps(normalized, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def load_df(path: str, dataset_name: str, id_column: str = None) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df["__dataset"] = dataset_name
    df["__source_file"] = Path(path).name

    if id_column and id_column in df.columns:
        df["unique_key"] = df[id_column].astype(str)
    else:
        df["unique_key"] = df.apply(row_hash, axis=1)

    return df


In [5]:
# Update these paths to your local copies
ABORTION_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/abortion_data-updated - new_abortion_related_subreddits_text_posts .csv"
MISCARRIAGE_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/miscarriage-data-updated - miscarriage_related_posts.csv"
HARASSMENT_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/sexual-harrassment-data-updated - RelevantByTitle.csv"

# History file (persists across runs)
HISTORY_PATH = Path("/home/ubuntu/TW_MultiLabel_SMP/jupyter-notebooks/selection_history.json")

# Desired counts per dataset
counts = {
    "abortion": 167,
    "miscarriage": 166,
    "harassment": 167
}

# Optional: if your CSVs have a post_id or id column
ID_COLUMN = None   # e.g. "post_id"


In [6]:
# Load CSVs
abortion_df = load_df(ABORTION_PATH, "abortion", ID_COLUMN)
miscarriage_df = load_df(MISCARRIAGE_PATH, "miscarriage", ID_COLUMN)
harassment_df = load_df(HARASSMENT_PATH, "harassment", ID_COLUMN)

# Load or initialize selection history
if HISTORY_PATH.exists():
    with open(HISTORY_PATH, "r", encoding="utf-8") as f:
        history = json.load(f)
else:
    history = {"used_keys": [], "runs": []}

used_keys = set(history.get("used_keys", []))

# Exclude previously used posts
def exclude_used(df):
    return df[~df["unique_key"].isin(used_keys)].copy()

ab_pool = exclude_used(abortion_df)
mi_pool = exclude_used(miscarriage_df)
sh_pool = exclude_used(harassment_df)

In [7]:
shortages = []
if len(ab_pool) < counts["abortion"]:
    shortages.append(f"abortion (need {counts['abortion']}, have {len(ab_pool)})")
if len(mi_pool) < counts["miscarriage"]:
    shortages.append(f"miscarriage (need {counts['miscarriage']}, have {len(mi_pool)})")
if len(sh_pool) < counts["harassment"]:
    shortages.append(f"harassment (need {counts['harassment']}, have {len(sh_pool)})")

if shortages:
    raise RuntimeError("Not enough fresh rows: " + "; ".join(shortages))

sample_ab = ab_pool.sample(n=counts["abortion"], replace=False, random_state=None)
sample_mi = mi_pool.sample(n=counts["miscarriage"], replace=False, random_state=None)
sample_sh = sh_pool.sample(n=counts["harassment"], replace=False, random_state=None)

sample_all = pd.concat([sample_ab, sample_mi, sample_sh], ignore_index=True)
sample_all = sample_all.sample(frac=1.0).reset_index(drop=True)  # shuffle


In [8]:
# Save timestamped CSV
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = Path(f"unique_sample_{ts}.csv")
sample_all.to_csv(out_path, index=False)

# Update history
new_keys = sample_all["unique_key"].tolist()
history["used_keys"].extend(new_keys)
history["runs"].append({
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "output_file": str(out_path),
    "counts": counts,
    "selected": len(new_keys)
})

with open(HISTORY_PATH, "w", encoding="utf-8") as f:
    json.dump(history, f, ensure_ascii=False, indent=2)

print(f"✅ Saved {len(sample_all)} posts to {out_path}")
print(f"Remaining after this run:")
print("  abortion:", len(ab_pool) - counts["abortion"])
print("  miscarriage:", len(mi_pool) - counts["miscarriage"])
print("  harassment:", len(sh_pool) - counts["harassment"])


✅ Saved 500 posts to unique_sample_20251016_174914.csv
Remaining after this run:
  abortion: 3576
  miscarriage: 154
  harassment: 4327


In [9]:
# Make a clean training label column (good for ML pipelines)
sample_all = sample_all.copy()
sample_all["label"] = sample_all["__dataset"]  # keep your original columns intact

# Create a training_sets folder
TRAIN_DIR = Path("training_sets")
TRAIN_DIR.mkdir(parents=True, exist_ok=True)

# Save a per-run training file (500 rows)
train_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
train_csv = TRAIN_DIR / f"train_{train_ts}.csv"
sample_all.to_csv(train_csv, index=False)

# Also keep a stable "latest" pointer you can reference in code
latest_csv = TRAIN_DIR / "train_latest.csv"
sample_all.to_csv(latest_csv, index=False)

print(f"✅ Saved training set (500 rows): {train_csv}")
print(f"🔁 Also updated: {latest_csv}")

# (Optional) Save per-class training files for class-specific experiments
PER_CLASS_DIR = TRAIN_DIR / f"per_class_{train_ts}"
PER_CLASS_DIR.mkdir(parents=True, exist_ok=True)

for cls in sample_all["label"].unique():
    out_cls = PER_CLASS_DIR / f"{cls}_train_{train_ts}.csv"
    sample_all[sample_all["label"] == cls].to_csv(out_cls, index=False)
    print(f"• Saved {cls} subset to: {out_cls}")

# (Optional) Keep a cumulative union of everything ever sampled (good for audit/repro)
CUMULATIVE_CSV = TRAIN_DIR / "all_selected_so_far.csv"
if CUMULATIVE_CSV.exists():
    prev = pd.read_csv(CUMULATIVE_CSV, low_memory=False)
    # Use unique_key to de-dup
    combined = pd.concat([prev, sample_all], ignore_index=True)
    combined = combined.drop_duplicates(subset=["unique_key"])
else:
    combined = sample_all

combined.to_csv(CUMULATIVE_CSV, index=False)
print(f"📚 Cumulative selected-so-far updated: {CUMULATIVE_CSV}")


✅ Saved training set (500 rows): training_sets/train_20251016_175203.csv
🔁 Also updated: training_sets/train_latest.csv
• Saved harassment subset to: training_sets/per_class_20251016_175203/harassment_train_20251016_175203.csv
• Saved abortion subset to: training_sets/per_class_20251016_175203/abortion_train_20251016_175203.csv
• Saved miscarriage subset to: training_sets/per_class_20251016_175203/miscarriage_train_20251016_175203.csv
📚 Cumulative selected-so-far updated: training_sets/all_selected_so_far.csv


In [10]:
sample_all.head(10)

,id,subreddit,title,selftext,created_utc,url,Tags,__dataset,__source_file,unique_key,label
0,n1i0pd,assault,"I’m a male that was coerced, defrauded, assaul...","Nearly 5 years ago, I was 20 years old, couldn...",2021-04-30 0:12:14,https://www.reddit.com/r/sexualassault/comment...,NaN,harassment,sexual-harrassment-data-updated - RelevantByTi...,9fa99569d92204435c178442157d70f1c03f37da8e1cc8...,harassment
1,1lb4r4g,Miscarriage,MMC in Australia. What now?,"Hi everyone, I'm sorry to be here and I'm sorr...",2025-06-14 9:34:24,https://www.reddit.com/r/Miscarriage/comments/...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,42a5ad9f1fd56ff15493ce3376a720c5bcdfa8f1844ee0...,abortion
2,1lkcz7z,Miscarriage,"8 week incomplete miscarriage, undecided next ...",TW: describe symptoms of what I've experienced...,2025-06-25 18:08:24,https://www.reddit.com/r/Miscarriage/comments/...,NaN,miscarriage,miscarriage-data-updated - miscarriage_related...,b022789aba45e433517c1ddd6d4100f422ad83fb1457f7...,miscarriage
3,aecuyf,metoo,Help with an answer,I have a friend who says it’s not right that a...,2019-01-09 23:33:42,https://www.reddit.com/r/meToo/comments/aecuyf...,NaN,harassment,sexual-harrassment-data-updated - RelevantByTi...,befd5c6bedf7e1f8f3e88f18cc3e2475a7c57df71559ee...,harassment
4,1lgksxs,BabyBumps,A Bit of Friend Drama…,"Buckle in, this will be a long one 😅\n\nMy bes...",2025-06-21 1:35:48,https://www.reddit.com/r/BabyBumps/comments/1l...,NaN,miscarriage,miscarriage-data-updated - miscarriage_related...,3e0ec7d610d0647e9d4b64f2859462d624ff1d8edbf069...,miscarriage
5,1le77ri,AmItheAsshole,WIBTA for telling my parents that my brother b...,I (22F) just moved back home from college afte...,2025-06-18 3:37:10,https://www.reddit.com/r/AmItheAsshole/comment...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,70ec4041f309dafa58fd6988633c1ac90e75a95133a117...,abortion
6,1loayzi,Pregnant,how many days post miscarriage did you find ou...,I had a miscarriage 19 days ago at 6 weeks (HC...,2025-06-30 15:47:18,https://www.reddit.com/r/pregnant/comments/1lo...,NaN,miscarriage,miscarriage-data-updated - miscarriage_related...,5ea1cbfb1f5c535391476ebf70cbdd0650f262cc945d65...,miscarriage
7,1799wqh,relationships,How to get over the fact that my boyfriend has...,"Hi,\nHow do I (22F) get over the fact that my ...",2023-10-16 16:21:08,https://www.reddit.com/r/relationships/comment...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,85ba52c790334a4ba496fe7bd3d80a414c7e17e9ac8fd9...,abortion
8,hxcjlb,metoo,My brothers.,"I wasn’t really how to start this post out, bu...",2020-07-24 23:38:41,https://www.reddit.com/r/meToo/comments/hxcjlb...,NaN,harassment,sexual-harrassment-data-updated - RelevantByTi...,22290fdeba53e578b7983aa13a6f3421ebedb870704464...,harassment
9,1ljds0c,Miscarriage,Grieving a chemical pregnancy,So I’m 19 and my boyfriend is 18. Last week I ...,2025-06-24 15:15:10,https://www.reddit.com/r/Miscarriage/comments/...,NaN,miscarriage,miscarriage-data-updated - miscarriage_related...,fd46eda5d36a256b511d132b6ecd21934e4f0c6f859fe6...,miscarriage
